In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from dataclasses import dataclass

@dataclass
class AlcabalaParams:
    uit: float = 5500.00          # UIT 2026 (S/ 5,500) [1](https://www.gob.pe/435-valor-de-la-uit-en-el-ano-2026)[2](https://busquedas.elperuano.pe/dispositivo/NL/2469116-1)
    tasa: float = 0.03            # 3% [3](https://www.sat.gob.pe/WebSiteV9/TributosMultas/ImpuestoAlcabala/Informacion)
    uit_inafectas: int = 10       # 10 UIT inafectas [3](https://www.sat.gob.pe/WebSiteV9/TributosMultas/ImpuestoAlcabala/Informacion)

def calc_alcabala(valor_transferencia: float,
                  valor_autovaluo: float | None = None,
                  params: AlcabalaParams = AlcabalaParams()) -> dict:
    """
    Calcula Impuesto de Alcabala (Perú) con regla general:
    Base = max(valor_transferencia, autovaluo) - 10*UIT
    Impuesto = max(Base, 0) * 3%
    [3](https://www.sat.gob.pe/WebSiteV9/TributosMultas/ImpuestoAlcabala/Informacion)
    """
    if valor_transferencia < 0:
        raise ValueError("El valor de transferencia no puede ser negativo.")

    if valor_autovaluo is None:
        valor_base = valor_transferencia
        usado = "transferencia"
    else:
        if valor_autovaluo < 0:
            raise ValueError("El autovalúo no puede ser negativo.")
        valor_base = max(valor_transferencia, valor_autovaluo)
        usado = "autovaluo" if valor_autovaluo > valor_transferencia else "transferencia"

    tramo_inafecto = params.uit_inafectas * params.uit
    base_imponible = max(valor_base - tramo_inafecto, 0.0)
    impuesto = base_imponible * params.tasa

    return {
        "valor_transferencia": valor_transferencia,
        "valor_autovaluo": valor_autovaluo,
        "valor_considerado": valor_base,
        "criterio_usado": usado,
        "uit": params.uit,
        "tramo_inafecto": tramo_inafecto,
        "base_imponible": base_imponible,
        "tasa": params.tasa,
        "alcabala": impuesto
    }

def money(x: float) -> str:
    return f"S/ {x:,.2f}"

def main():
    print("=== Calculadora de Impuesto de Alcabala (Perú) ===")
    print("Regla general: (max(Transferencia, Autovalúo) - 10*UIT) * 3%")  # [3](https://www.sat.gob.pe/WebSiteV9/TributosMultas/ImpuestoAlcabala/Informacion)
    print()

    monto_depa = float(input("Monto del depa (valor de transferencia) en soles: ").replace(",", ""))
    autovaluo_in = input("Autovalúo (opcional, Enter para omitir): ").strip().replace(",", "")

    autovaluo = float(autovaluo_in) if autovaluo_in else None

    # Si quieres cambiar UIT/año o tasa:
    # params = AlcabalaParams(uit=5500.0, tasa=0.03, uit_inafectas=10)
    params = AlcabalaParams()

    r = calc_alcabala(monto_depa, autovaluo, params)

    print("\n--- Resultado ---")
    print(f"Valor transferencia: {money(r['valor_transferencia'])}")
    if r["valor_autovaluo"] is not None:
        print(f"Autovalúo:          {money(r['valor_autovaluo'])}")
    print(f"Valor considerado:  {money(r['valor_considerado'])}  (usó: {r['criterio_usado']})")
    print(f"Tramo inafecto:     {money(r['tramo_inafecto'])}  (10 UIT; UIT={money(r['uit'])})")
    print(f"Base imponible:     {money(r['base_imponible'])}")
    print(f"Tasa:              {r['tasa']*100:.2f}%")
    print(f"ALCABALA a pagar:   {money(r['alcabala'])}")

if __name__ == "__main__":
    main()


=== Calculadora de Impuesto de Alcabala (Perú) ===
Regla general: (max(Transferencia, Autovalúo) - 10*UIT) * 3%


--- Resultado ---
Valor transferencia: S/ 10,000.00
Valor considerado:  S/ 10,000.00  (usó: transferencia)
Tramo inafecto:     S/ 55,000.00  (10 UIT; UIT=S/ 5,500.00)
Base imponible:     S/ 0.00
Tasa:              3.00%
ALCABALA a pagar:   S/ 0.00
